# 🎬 Sentiment Analysis Tool for Movie Reviews

**Goal:** read a piece of text (a movie review, tweet, etc.) and automatically classify it as **negative**, **neutral**, or **positive** using machine learning.

**Pipeline**
1. Load a real movie-review dataset (Stanford Sentiment Treebank, via Hugging Face)
2. Explore the data
3. Clean the text
4. Train a baseline model (TF-IDF + Logistic Regression) and compare a few classifiers
5. Evaluate (accuracy, F1, confusion matrix) and do error analysis
6. Build a `predict_sentiment()` tool + interactive widget
7. Save the model
8. *Upgrade:* compare against a pretrained transformer (RoBERTa)

> Run cells top to bottom. For step 8, switch to a GPU: **Runtime → Change runtime type → T4 GPU**.

## 1. Setup

In [ ]:
!pip install -q datasets scikit-learn matplotlib seaborn joblib transformers

In [ ]:
import re
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2. Load the data

IMDB-style datasets only have positive/negative, so there's no neutral class. We use **SST-5** (Stanford Sentiment Treebank, 5 labels) and collapse it to 3:

| Original label | Meaning | Our label |
|---|---|---|
| 0, 1 | very negative, negative | **negative** |
| 2 | neutral | **neutral** |
| 3, 4 | positive, very positive | **positive** |

In [ ]:
ds = load_dataset("SetFit/sst5")
print(ds)

LABELS = ["negative", "neutral", "positive"]

def to_3class(label):
    if label in (0, 1):
        return 0      # negative
    if label == 2:
        return 1      # neutral
    return 2          # positive

train_df = ds["train"].to_pandas()
val_df   = ds["validation"].to_pandas()
test_df  = ds["test"].to_pandas()

for df in (train_df, val_df, test_df):
    df["sentiment"] = df["label"].map(to_3class)
    df["sentiment_name"] = df["sentiment"].map(lambda i: LABELS[i])

train_df[["text", "sentiment_name"]].sample(5, random_state=RANDOM_STATE)

## 3. Explore the data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=train_df, x="sentiment_name", order=LABELS, ax=axes[0])
axes[0].set_title("Class distribution (train)")
axes[0].set_xlabel("")

train_df["n_words"] = train_df["text"].str.split().str.len()
sns.histplot(train_df["n_words"], bins=40, ax=axes[1])
axes[1].set_title("Review length (words)")

plt.tight_layout()
plt.show()

print(train_df["sentiment_name"].value_counts(normalize=True).round(3))
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

## 4. Clean the text

Light cleaning only. We deliberately **keep stopwords** like *not*, *no*, *never* — removing them would flip the meaning of "not good".

In [ ]:
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)   # URLs
    text = re.sub(r"@\w+", " ", text)                    # @mentions (tweets)
    text = re.sub(r"#", "", text)                        # keep hashtag word, drop '#'
    text = re.sub(r"<.*?>", " ", text)                   # HTML tags
    text = re.sub(r"[^a-z0-9!?'\s]", " ", text)          # keep letters, digits, ! ? '
    text = re.sub(r"\s+", " ", text).strip()
    return text

for df in (train_df, val_df, test_df):
    df["clean"] = df["text"].apply(clean_text)

print(train_df["text"].iloc[0])
print(train_df["clean"].iloc[0])

## 5. Train and compare models

TF-IDF turns each review into a vector of weighted word/bigram counts. Then we try three classic classifiers and compare them on the **validation** set using **macro F1** (treats all 3 classes equally, better than accuracy when classes are imbalanced).

In [ ]:
def make_pipeline(clf):
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("clf", clf),
    ])

candidates = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", C=2.0),
    "Linear SVM":          LinearSVC(class_weight="balanced"),
    "Naive Bayes":         MultinomialNB(),
}

results = []
fitted = {}
for name, clf in candidates.items():
    pipe = make_pipeline(clf)
    pipe.fit(train_df["clean"], train_df["sentiment"])
    preds = pipe.predict(val_df["clean"])
    results.append({
        "model": name,
        "accuracy": accuracy_score(val_df["sentiment"], preds),
        "macro_f1": f1_score(val_df["sentiment"], preds, average="macro"),
    })
    fitted[name] = pipe

pd.DataFrame(results).sort_values("macro_f1", ascending=False).round(4)

## 6. Final evaluation on the test set

We use Logistic Regression as the final model because it also gives **probabilities** (confidence scores), which the tool needs. Test data is only touched here, once.

In [ ]:
model = fitted["Logistic Regression"]

test_preds = model.predict(test_df["clean"])
print(classification_report(test_df["sentiment"], test_preds, target_names=LABELS, digits=3))

cm = confusion_matrix(test_df["sentiment"], test_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion matrix (test)")
plt.show()

### What the model learned: most influential words

In [ ]:
tfidf = model.named_steps["tfidf"]
clf = model.named_steps["clf"]
features = np.array(tfidf.get_feature_names_out())

for i, label in enumerate(LABELS):
    top = np.argsort(clf.coef_[i])[-12:][::-1]
    print(f"{label.upper():>9}: {', '.join(features[top])}")

## 7. Error analysis

Look at what the model gets wrong. This is where you learn the most.

In [ ]:
errors = test_df.assign(pred=[LABELS[p] for p in test_preds])
errors = errors[errors["sentiment_name"] != errors["pred"]]
print(f"{len(errors)} misclassified out of {len(test_df)} ({len(errors)/len(test_df):.1%})\n")

pd.set_option("display.max_colwidth", 140)
errors[["text", "sentiment_name", "pred"]].sample(8, random_state=RANDOM_STATE)

## 8. The tool: `predict_sentiment()`

In [ ]:
def predict_sentiment(texts, model=model):
    """Classify one string or a list of strings. Returns a DataFrame."""
    if isinstance(texts, str):
        texts = [texts]
    cleaned = [clean_text(t) for t in texts]
    probs = model.predict_proba(cleaned)
    preds = probs.argmax(axis=1)
    out = pd.DataFrame({
        "text": texts,
        "sentiment": [LABELS[p] for p in preds],
        "confidence": probs.max(axis=1).round(3),
    })
    for i, label in enumerate(LABELS):
        out[f"p_{label}"] = probs[:, i].round(3)
    return out

samples = [
    "An absolute masterpiece. The acting and the score were breathtaking!",
    "Boring, predictable, and way too long. I want my two hours back.",
    "The movie was released in 2019 and runs about 110 minutes.",
    "Not bad, but not great either.",
    "I wanted to love it, but the second half falls apart.",
]
predict_sentiment(samples)

### Interactive widget

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

box = widgets.Textarea(placeholder="Type or paste a movie review / tweet...", layout=widgets.Layout(width="90%", height="90px"))
btn = widgets.Button(description="Analyze", button_style="primary")
out = widgets.Output()

def on_click(_):
    with out:
        clear_output()
        if box.value.strip():
            display(predict_sentiment(box.value))

btn.on_click(on_click)
display(box, btn, out)

## 9. Save the model

In [ ]:
joblib.dump({"model": model, "labels": LABELS}, "sentiment_model.joblib")

# To reload later (or in another notebook / app):
# bundle = joblib.load("sentiment_model.joblib")
# model, LABELS = bundle["model"], bundle["labels"]

# Download to your computer from Colab:
# from google.colab import files
# files.download("sentiment_model.joblib")

## 10. Upgrade: pretrained transformer

Classic TF-IDF models see words, not context. A pretrained transformer understands things like sarcasm and negation much better. This model already outputs negative / neutral / positive. **Use a GPU runtime** for speed.

We compare on a 500-review sample of the test set so it runs quickly.

In [ ]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
roberta = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=device,
    truncation=True,
    max_length=256,
)

sample = test_df.sample(500, random_state=RANDOM_STATE)
rob_preds = [r["label"].lower() for r in roberta(sample["text"].tolist(), batch_size=32)]
lr_preds = [LABELS[p] for p in model.predict(sample["clean"])]

print("TF-IDF + LogReg  accuracy:", round(accuracy_score(sample["sentiment_name"], lr_preds), 3),
      "| macro F1:", round(f1_score(sample["sentiment_name"], lr_preds, average="macro"), 3))
print("RoBERTa          accuracy:", round(accuracy_score(sample["sentiment_name"], rob_preds), 3),
      "| macro F1:", round(f1_score(sample["sentiment_name"], rob_preds, average="macro"), 3))

In [ ]:
roberta(samples)

## 11. Next steps

- Fine-tune a transformer (DistilBERT) on SST-5 yourself instead of using a pretrained one
- Add a tweet dataset (`tweet_eval`, `sentiment` config) and test how the movie-trained model transfers
- Try sarcasm / negation test cases and document where each model fails
- Wrap `predict_sentiment()` in a Gradio or Streamlit app and deploy it (Hugging Face Spaces is free)
- Write up the results: baseline vs. transformer, with the confusion matrices